# U4-01 · De la neurona a Keras: perceptrón, retropropagación y optimizadores

*Especialista en Inteligencia Artificial (IFCD107) · Unidad 4*

Ejecuta cada celda con **Mayús + Intro** (en Colab, también con ▶). Con GPU (*Entorno de ejecución → Cambiar tipo de entorno de ejecución*) las redes se entrenan mucho más deprisa.

> Datos: Koklu, M. y Ozkan, I. A. (2020). *Computers and Electronics in Agriculture*, 174, 105507. https://doi.org/10.1016/j.compag.2020.105507 · Dry Bean, UCI, CC BY 4.0, https://doi.org/10.24432/C50S4B
> Datos: Fanaee-T, H. y Gama, J. (2014). Event labeling combining ensemble detectors and background knowledge. *Progress in Artificial Intelligence*, 2, 113–127. Bike Sharing, UCI, CC BY 4.0, https://doi.org/10.24432/C5W894

In [ ]:
# Descarga los datos y módulos auxiliares (solo la primera vez)
import os, urllib.request
for f in ["judias.csv", "keras_judias.py", "bicis.py", "hour.csv"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve("https://machinelearning-formacion.github.io/ifcd107/recursos/datos/u4/" + f, f)

## 4.1 · De la neurona a la red

`m1_neurona.py`

In [ ]:
import numpy as np
import pandas as pd

j = pd.read_csv("judias.csv").drop_duplicates()
j = j[j.Class.isin(["SEKER", "DERMASON"])]
X = j[["Area", "roundness"]].to_numpy()
X = (X - X.mean(0)) / X.std(0)                # escalar
y = (j.Class == "SEKER").to_numpy().astype(int)
rng = np.random.default_rng(0)
w, b = np.zeros(2), 0.0
for epoca in range(1, 6):          # perceptrón de Rosenblatt
    errores = 0
    for i in rng.permutation(len(y)):
        pred = int(X[i] @ w + b > 0)
        if pred != y[i]:                      # solo aprende si falla
            w += (y[i] - pred) * X[i]; b += y[i] - pred
            errores += 1
    acierto = ((X @ w + b > 0) == y).mean()
    print(f"época {epoca}: correcciones {errores:4d},"
          f" acierto {acierto:.3f}")
print("pesos:", w.round(2), "sesgo:", round(b, 2))

`m9_xor.py`

In [ ]:
import numpy as np
import keras

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype="float32")
y = np.array([0, 1, 1, 0])                # o exclusivo
for ocultas in [0, 2, 4]:
    keras.utils.set_random_seed(3)
    capas = [keras.Input((2,))]
    if ocultas:
        capas.append(keras.layers.Dense(ocultas, activation="tanh"))
    m = keras.Sequential(capas + [keras.layers.Dense(
        1, activation="sigmoid")])
    m.compile(keras.optimizers.Adam(0.05), "binary_crossentropy")
    m.fit(X, y, epochs=500, verbose=0)
    p = m.predict(X, verbose=0).ravel()
    print(f"{ocultas} neuronas ocultas: probabilidades {p.round(2)}"
          f" -> aciertos {((p > 0.5) == y).sum()} de 4")

`m2_retro.py`

In [ ]:
import numpy as np
import pandas as pd

j = pd.read_csv("judias.csv").drop_duplicates().sample(
    frac=1, random_state=0)
X = j.drop(columns="Class").to_numpy()
X = (X - X.mean(0)) / X.std(0)
clases = sorted(j.Class.unique())
Y = np.eye(7)[j.Class.map(clases.index)]     # codificación one-hot
Xe, Ye, Xp, Yp = X[:10000], Y[:10000], X[10000:], Y[10000:]
rng = np.random.default_rng(0)
W1 = rng.normal(0, np.sqrt(2 / 16), (16, 32)); b1 = np.zeros(32)
W2 = rng.normal(0, np.sqrt(1 / 32), (32, 7)); b2 = np.zeros(7)
def adelante(X):
    Z = X @ W1 + b1; H = np.maximum(Z, 0)     # capa oculta ReLU
    S = H @ W2 + b2
    P = np.exp(S - S.max(1, keepdims=True))
    return Z, H, P / P.sum(1, keepdims=True)  # softmax
eta = 0.1
for epoca in range(1, 31):
    for k in range(0, 10000, 100):            # lotes de 100
        x, t = Xe[k:k + 100], Ye[k:k + 100]
        Z, H, P = adelante(x)
        dS = (P - t) / len(x)                 # derivada de la pérdida
        dW2 = H.T @ dS; db2 = dS.sum(0)
        dZ = (dS @ W2.T) * (Z > 0)            # regla de la cadena
        dW1 = x.T @ dZ; db1 = dZ.sum(0)
        W1 -= eta * dW1; b1 -= eta * db1
        W2 -= eta * dW2; b2 -= eta * db2
    if epoca in (1, 5, 10, 30):
        P = adelante(Xe)[2]
        perdida = -np.mean(np.log(P[Ye == 1]))
        acierto = (adelante(Xp)[2].argmax(1) == Yp.argmax(1)).mean()
        print(f"época {epoca:2d}: pérdida {perdida:.3f},"
              f" acierto prueba {acierto:.3f}")
print("parámetros:", W1.size + b1.size + W2.size + b2.size)

`m6_autodiff.py`

In [ ]:
import numpy as np
import tensorflow as tf

rng = np.random.default_rng(0)
x = rng.normal(size=(4, 3)).astype("float32")
t = np.eye(2, dtype="float32")[[0, 1, 1, 0]]
W1 = tf.Variable(rng.normal(size=(3, 5)).astype("float32"))
W2 = tf.Variable(rng.normal(size=(5, 2)).astype("float32"))
with tf.GradientTape() as cinta:          # graba las operaciones
    z = x @ W1; h = tf.nn.relu(z)
    p = tf.nn.softmax(h @ W2)
    L = -tf.reduce_mean(tf.reduce_sum(t * tf.math.log(p), 1))
g1, g2 = cinta.gradient(L, [W1, W2])     # derivadas automáticas
dS = (p.numpy() - t) / 4                  # las mismas, a mano
m2 = h.numpy().T @ dS
m1 = x.T @ ((dS @ W2.numpy().T) * (z.numpy() > 0))
print(f"pérdida {float(L):.4f}")
print("diferencia máxima W2:", float(np.abs(g2.numpy() - m2).max()))
print("diferencia máxima W1:", float(np.abs(g1.numpy() - m1).max()))

## 4.2 · Redes con Keras

`m3_keras.py`

In [ ]:
import keras
from keras_judias import X_e, X_p, y_e, y_p

keras.utils.set_random_seed(0)
modelo = keras.Sequential([
    keras.Input(shape=(16,)),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(7, activation="softmax")])
modelo.compile(optimizer=keras.optimizers.SGD(0.1),
               loss="sparse_categorical_crossentropy",
               metrics=["accuracy"])
for c in modelo.layers:
    print(f"{c.name:<8} salida {str(c.output.shape):<12}"
          f" parámetros {c.count_params()}")
h = modelo.fit(X_e, y_e, epochs=30, batch_size=100,
               validation_split=0.2, verbose=0)
for e in (0, 4, 9, 29):
    print(f"época {e + 1:2d}: pérdida {h.history['loss'][e]:.3f}"
          f"  acierto val. {h.history['val_accuracy'][e]:.3f}")
perdida, acierto = modelo.evaluate(X_p, y_p, verbose=0)
print(f"prueba: acierto {acierto:.3f}")

`m4_optimizadores.py`

In [ ]:
import keras
from keras_judias import X_e, y_e

def entrenar(opt):
    keras.utils.set_random_seed(0)
    m = keras.Sequential([keras.Input(shape=(16,)),
                          keras.layers.Dense(64, activation="relu"),
                          keras.layers.Dense(64, activation="relu"),
                          keras.layers.Dense(7,
                                             activation="softmax")])
    m.compile(optimizer=opt, loss="sparse_categorical_crossentropy")
    h = m.fit(X_e, y_e, epochs=20, batch_size=64, verbose=0)
    return h.history["loss"]
for nombre, opt in [
        ("SGD 0,01", keras.optimizers.SGD(0.01)),
        ("SGD 0,01 + momento",
         keras.optimizers.SGD(0.01, momentum=0.9)),
        ("RMSprop 0,001", keras.optimizers.RMSprop(0.001)),
        ("Adam 0,001", keras.optimizers.Adam(0.001)),
        ("SGD 3 (demasiado alta)", keras.optimizers.SGD(3.0))]:
    p = entrenar(opt)
    print(f"{nombre:<23} pérdida: época 1 {p[0]:6.3f},"
          f" 5 {p[4]:6.3f}, 20 {p[19]:6.3f}")

`m7_inicio_lote.py`

In [ ]:
import time
import keras
from keras_judias import X_e, X_p, y_e, y_p

def red(inicio="he_normal", lote=64, epocas=20):
    keras.utils.set_random_seed(0)
    m = keras.Sequential(
        [keras.Input(shape=(16,))] + [
        keras.layers.Dense(64, activation="relu",
                           kernel_initializer=inicio)
        for _ in range(2)]
        + [keras.layers.Dense(7, activation="softmax")])
    m.compile("adam", "sparse_categorical_crossentropy",
              metrics=["accuracy"])
    t = time.perf_counter()
    m.fit(X_e, y_e, epochs=epocas, batch_size=lote, verbose=0)
    return m.evaluate(X_p, y_p, verbose=0)[1], time.perf_counter() - t
print(f"pesos iniciales a cero:  acierto {red('zeros')[0]:.3f}")
print(f"pesos aleatorios (He):   acierto {red()[0]:.3f}")
for lote in [16, 256, 2048]:
    a, s = red(lote=lote)
    print(f"lote {lote:4d}: acierto {a:.3f}, {s:4.1f} s"
          f" ({len(X_e) // lote + 1} pasos por época)")

`m5_gradiente.py`

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
x = rng.normal(size=(500, 64))
for nombre, f, df, escala in [
        ("sigmoide", lambda z: 1 / (1 + np.exp(-z)),
         lambda z: np.exp(-z) / (1 + np.exp(-z)) ** 2, 1.0),
        ("ReLU", lambda z: np.maximum(z, 0),
         lambda z: (z > 0).astype(float), 2.0)]:
    W = [rng.normal(0, np.sqrt(escala / 64), (64, 64))
         for _ in range(10)]
    h, Z = x, []
    for w in W:                             # hacia delante
        Z.append(h @ w); h = f(Z[-1])
    g = np.ones_like(h)                     # hacia atrás
    tam = []
    for w, z in zip(W[::-1], Z[::-1]):
        g = g * df(z); tam.append(np.abs(g).mean()); g = g @ w.T
    print(f"{nombre:<9} tamaño medio del gradiente, capa 10 -> 1:",
          " ".join(f"{t:.0e}" for t in tam[::3]))

`m8_tabular.py`

In [ ]:
import numpy as np
import keras
from keras import layers
from bicis import X, y, ent, pru

num = ["temp_c", "humedad", "viento_kmh", "año", "laborable",
       "festivo"]
Xn = X[num].to_numpy("float32")
Xn = (Xn - Xn[ent].mean(0)) / Xn[ent].std(0)
hora = X.hora.to_numpy(); dia = X.dia_semana.to_numpy()
tiempo = X.tiempo.to_numpy(); mes = X.mes.to_numpy()
def entrada(n, nombre):                   # categoría -> incrustación
    e = keras.Input((1,), name=nombre)
    return e, layers.Flatten()(layers.Embedding(n, 4)(e))
keras.utils.set_random_seed(0)
eh, vh = entrada(24, "hora"); ed, vd = entrada(7, "dia")
et, vt = entrada(4, "tiempo"); em, vm = entrada(13, "mes")
en = keras.Input((len(num),), name="num")
x = layers.Concatenate()([vh, vd, vt, vm, en])
for _ in range(3):
    x = layers.Dense(128, activation="relu")(x)
sal = layers.Dense(1, activation="exponential")(x)   # media > 0
m = keras.Model([eh, ed, et, em, en], sal)
m.compile(keras.optimizers.Adam(1e-3), "poisson")
datos = lambda s: [hora[s], dia[s], tiempo[s], mes[s], Xn[s]]
m.fit(datos(ent), y[ent].to_numpy("float32"), epochs=60,
      batch_size=128, verbose=0)
p = m.predict(datos(pru), verbose=0).ravel()
print(f"red con incrustaciones: EAM {np.abs(y[pru] - p).mean():.1f}")

**Prueba.** En `m2_retro.py`, cambia la capa oculta a 8 y a 128 neuronas. ¿Cambia el acierto de prueba?

**Ejercicio.** En `m9_xor.py`, prueba otras semillas con 2 neuronas ocultas. ¿Cuántas resuelven el o exclusivo?